In [1]:
pip install XlsxWriter

Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from statistics import NormalDist

import numpy as np
import pandas as pd
INPUT_FILE = Path.home() / "Downloads" / "DDAIC_ass_data_2026 (1).xlsx"
OUTPUT_FILE = Path.home() / "Downloads" / "DDAIC_code_reproducible_output.xlsx"

TRAIN_START = "2015-01-01"
TRAIN_END = "2017-12-31"
TEST_START = "2018-01-01"
TEST_END = "2018-06-30"

TOTAL_FILL_RATE_TARGET = 0.98
MIN_FILL_RATE_GRID = [0.50, 0.60, 0.70, 0.80]

NORMAL = NormalDist()

In [3]:
xls = pd.ExcelFile(INPUT_FILE)
xls.sheet_names
article_preview = pd.read_excel(INPUT_FILE, sheet_name="ArticleData")
article_preview.head()

,code,productFamily,productFamilyE,target SL,MOQ,salesPrice,LT (days)
0,1111,Puzzels,Puzzles,0.98,1,20.56,3
1,1126,Tafelpapier,Table paper,0.98,1008,1.44,7
2,1127,Tafelpapier,Table paper,0.95,1008,1.71,7
3,1129,Keukengereedschap,Kitchen tools,0.98,520,16.19,3
4,1131,Fiets accessoires,Bicycle accessories,0.98,960,5.19,7


In [4]:
article_preview = pd.read_excel(INPUT_FILE, sheet_name="ArticleData")
display(article_preview.head())
print(article_preview.columns.tolist())

,code,productFamily,productFamilyE,target SL,MOQ,salesPrice,LT (days)
0,1111,Puzzels,Puzzles,0.98,1,20.56,3
1,1126,Tafelpapier,Table paper,0.98,1008,1.44,7
2,1127,Tafelpapier,Table paper,0.95,1008,1.71,7
3,1129,Keukengereedschap,Kitchen tools,0.98,520,16.19,3
4,1131,Fiets accessoires,Bicycle accessories,0.98,960,5.19,7


['code', 'productFamily', 'productFamilyE', 'target SL', 'MOQ', 'salesPrice', 'LT (days)']


In [5]:
def parse_demand_dates(date_values: pd.Series) -> pd.Series:
    """
    Robustly parses demand date columns.

    Handles:
    - 20150301
    - 20150301.0
    - Excel serial dates
    - normal date strings
    - actual Timestamp values
    """

    parsed_dates = []

    for value in date_values:
        if pd.isna(value):
            parsed_dates.append(pd.NaT)
            continue

        if isinstance(value, pd.Timestamp):
            parsed_dates.append(pd.to_datetime(value))
            continue

        text = str(value).strip()

        if text.endswith(".0"):
            text = text[:-2]

        date = pd.to_datetime(text, format="%Y%m%d", errors="coerce")

        if pd.isna(date):
            date = pd.to_datetime(text, errors="coerce")

        if pd.isna(date):
            try:
                number = float(text)

                if 20000 <= number <= 60000:
                    date = pd.to_datetime(
                        number,
                        unit="D",
                        origin="1899-12-30",
                        errors="coerce",
                    )
            except Exception:
                pass

        parsed_dates.append(date)

    return pd.Series(parsed_dates, index=date_values.index)

In [6]:
def prepare_data() -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Reads ArticleData and Demand data.

    Output:
        article:
            SKU, Q, LeadTime, UnitPrice

        demand:
            SKU, Date, Demand
    """

    if not INPUT_FILE.exists():
        raise FileNotFoundError(
            f"Could not find the input file here:\n{INPUT_FILE}\n\n"
            "Put the Excel file in your Downloads folder or change INPUT_FILE."
        )

    # ============================================================
    # 1. Article data
    # ============================================================
    article = pd.read_excel(INPUT_FILE, sheet_name="ArticleData")

    print("ArticleData columns:")
    print(article.columns.tolist())

    article = article.rename(
        columns={
            "code": "SKU",
            "MOQ": "Q",
            "salesPrice": "UnitPrice",
            "LT (days)": "LeadTime",
        }
    )

    required_cols = ["SKU", "Q", "UnitPrice", "LeadTime"]
    missing = [col for col in required_cols if col not in article.columns]

    if missing:
        raise ValueError(
            f"Missing columns in ArticleData after renaming: {missing}\n"
            f"Available columns are: {article.columns.tolist()}"
        )

    article["SKU"] = (
        pd.to_numeric(article["SKU"], errors="coerce")
        .astype("Int64")
        .astype(str)
    )

    article = article[article["SKU"] != "<NA>"].copy()

    article["Q"] = (
        pd.to_numeric(article["Q"], errors="coerce")
        .fillna(1)
        .clip(lower=1)
    )

    article["LeadTime"] = (
        pd.to_numeric(article["LeadTime"], errors="coerce")
        .fillna(1)
        .clip(lower=1)
        .round()
        .astype(int)
    )

    article["UnitPrice"] = (
        pd.to_numeric(article["UnitPrice"], errors="coerce")
        .fillna(0)
        .clip(lower=0)
    )

    article = article[["SKU", "Q", "LeadTime", "UnitPrice"]].copy()

    # ============================================================
    # 2. Demand data
    # ============================================================
    raw = pd.read_excel(INPUT_FILE, sheet_name="Demand data", header=None)

    print("\nFirst rows of raw Demand data:")
    display(raw.iloc[:8, :12])

    # Find the row that contains the date headers.
    # We search the first 15 rows and pick the row with the most valid dates.
    best_row = None
    best_count = -1
    best_dates = None

    for row_number in range(min(15, len(raw))):
        candidate_dates = parse_demand_dates(raw.iloc[row_number, 2:])
        valid_count = candidate_dates.notna().sum()

        print(f"Row {row_number}: detected {valid_count} date columns")

        if valid_count > best_count:
            best_count = valid_count
            best_row = row_number
            best_dates = candidate_dates

    print("\nDetected date row:", best_row)
    print("Number of detected date columns:", best_count)

    if best_count <= 1:
        raise ValueError(
            "The script detected one or zero date columns. "
            "The demand sheet structure is different than expected. "
            "Check the displayed raw Demand data above."
        )

    date_columns = best_dates

    # Data starts below the detected date row.
    demand_wide = raw.iloc[best_row + 1:, :].copy()

    demand_wide = demand_wide.rename(
        columns={
            0: "SKU",
            1: "productFamily",
        }
    )

    demand_wide = demand_wide.dropna(subset=["SKU"])

    demand_wide["SKU"] = (
        pd.to_numeric(demand_wide["SKU"], errors="coerce")
        .astype("Int64")
        .astype(str)
    )

    demand_wide = demand_wide[demand_wide["SKU"] != "<NA>"].copy()

    valid_date_mask = date_columns.notna()

    demand_value_columns = list(raw.columns[2:][valid_date_mask])
    valid_dates = list(date_columns[valid_date_mask])

    demand_wide = demand_wide[["SKU"] + demand_value_columns]

    rename_map = dict(zip(demand_value_columns, valid_dates))
    demand_wide = demand_wide.rename(columns=rename_map)

    demand = demand_wide.melt(
        id_vars="SKU",
        var_name="Date",
        value_name="Demand",
    )

    demand["Date"] = pd.to_datetime(demand["Date"])

    demand["Demand"] = (
        pd.to_numeric(demand["Demand"], errors="coerce")
        .fillna(0)
        .clip(lower=0)
    )

    # Remove rows where the Date could not be parsed.
    demand = demand.dropna(subset=["Date"]).copy()

    print("\nPrepared ArticleData:")
    print(article.shape)
    display(article.head())

    print("\nPrepared Demand data:")
    print(demand.shape)
    display(demand.head())

    print("\nDemand date range:")
    print("Min date:", demand["Date"].min())
    print("Max date:", demand["Date"].max())
    print("Unique dates:", demand["Date"].nunique())

    print("\nDemand by year:")
    display(demand.groupby(demand["Date"].dt.year)["Demand"].sum())

    return article, demand

In [7]:
article, demand = prepare_data()

print("Article shape:", article.shape)
print("Demand shape:", demand.shape)

print("Min date:", demand["Date"].min())
print("Max date:", demand["Date"].max())
print("Unique dates:", demand["Date"].nunique())

display(demand.head())
display(demand.tail())

ArticleData columns:
['code', 'productFamily', 'productFamilyE', 'target SL', 'MOQ', 'salesPrice', 'LT (days)']

First rows of raw Demand data:


,0,1,2,3,4,5,6,7,8,9,10,11
0,code,productFamily,Demand per date,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,20150301,20150302.0,20150303.0,20150304.0,20150305.0,20150306.0,20150307.0,20150308.0,20150309.0,20150310.0
2,1111,Puzzels,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1126,Tafelpapier,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1127,Tafelpapier,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,1129,Keukengereedschap,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,1131,Fiets accessoires,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,1134,Strijkplanken,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Row 0: detected 0 date columns
Row 1: detected 1218 date columns
Row 2: detected 0 date columns
Row 3: detected 4 date columns
Row 4: detected 0 date columns
Row 5: detected 0 date columns
Row 6: detected 0 date columns
Row 7: detected 0 date columns
Row 8: detected 0 date columns
Row 9: detected 0 date columns
Row 10: detected 0 date columns
Row 11: detected 0 date columns
Row 12: detected 0 date columns
Row 13: detected 0 date columns
Row 14: detected 0 date columns

Detected date row: 1
Number of detected date columns: 1218

Prepared ArticleData:
(1723, 4)


,SKU,Q,LeadTime,UnitPrice
0,1111,1,3,20.56
1,1126,1008,7,1.44
2,1127,1008,7,1.71
3,1129,520,3,16.19
4,1131,960,7,5.19



Prepared Demand data:
(2098614, 3)


...


Demand date range:
Min date: 2015-03-01 00:00:00
Max date: 2018-06-30 00:00:00
Unique dates: 1218

Demand by year:


Date
2015    32837420
2016    41428383
2017    44812624
2018    20676816
Name: Demand, dtype: int64

Article shape: (1723, 4)
Demand shape: (2098614, 3)
Min date: 2015-03-01 00:00:00
Max date: 2018-06-30 00:00:00
Unique dates: 1218


,SKU,Date,Demand
0,1111,2015-03-01,0
1,1126,2015-03-01,0
2,1127,2015-03-01,0
3,1129,2015-03-01,0
4,1131,2015-03-01,0


,SKU,Date,Demand
2098609,7239,2018-06-30,65
2098610,7240,2018-06-30,1
2098611,7242,2018-06-30,42
2098612,7244,2018-06-30,0
2098613,7246,2018-06-30,0


In [8]:
print(demand["Date"].min())
print(demand["Date"].max())

2015-03-01 00:00:00
2018-06-30 00:00:00


In [9]:
def make_daily_panel(demand: pd.DataFrame) -> pd.DataFrame:
    """
    Ensures each SKU has one row for every day from 2015-01-01 to 2018-06-30.
    Missing demand days are filled with zero.
    """

    all_skus = demand["SKU"].unique()
    all_dates = pd.date_range(TRAIN_START, TEST_END, freq="D")

    idx = pd.MultiIndex.from_product(
        [all_skus, all_dates],
        names=["SKU", "Date"],
    )

    daily = (
        demand.groupby(["SKU", "Date"], as_index=False)["Demand"]
        .sum()
        .set_index(["SKU", "Date"])
        .reindex(idx, fill_value=0)
        .reset_index()
    )

    return daily

In [10]:
daily = make_daily_panel(demand)

print(daily.shape)
display(daily.head())
print(daily["Date"].min())
print(daily["Date"].max())
print(daily["SKU"].nunique())

(2200271, 3)


,SKU,Date,Demand
0,1111,2015-01-01,0
1,1111,2015-01-02,0
2,1111,2015-01-03,0
3,1111,2015-01-04,0
4,1111,2015-01-05,0


2015-01-01 00:00:00
2018-06-30 00:00:00
1723


In [11]:
@dataclass
class ForecastResult:
    method: str
    sku: str
    forecast_daily: float
    actual_test: float
    forecast_test: float
    abs_error: float
    squared_error: float

In [12]:
def ses_forecast(train: np.ndarray, alpha: float) -> float:
    """
    Simple Exponential Smoothing.

    Formula:
        F_{t+1} = alpha * D_t + (1 - alpha) * F_t
    """

    if len(train) == 0:
        return 0.0

    level = float(train[0])

    for y in train[1:]:
        level = alpha * float(y) + (1 - alpha) * level

    return max(level, 0.0)

In [13]:
def croston_forecast(train: np.ndarray, alpha: float = 0.1) -> float:
    """
    Croston's method for intermittent demand.

    Forecast = estimated demand size / estimated interval
    """

    positive_positions = np.flatnonzero(train > 0)

    if len(positive_positions) == 0:
        return 0.0

    first = positive_positions[0]

    z = float(train[first])
    p = float(first + 1)
    last = first

    for pos in positive_positions[1:]:
        interval = pos - last

        z = alpha * float(train[pos]) + (1 - alpha) * z
        p = alpha * float(interval) + (1 - alpha) * p

        last = pos

    if p <= 0:
        return 0.0

    return max(z / p, 0.0)

In [14]:
example = np.array([0, 0, 3, 0, 0, 5, 0, 2])

print("SES:", ses_forecast(example, 0.2))
print("Croston:", croston_forecast(example, 0.1))

SES: 1.2366080000000004
Croston: 1.0620689655172415


In [15]:
def evaluate_forecasts(
    daily: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Evaluates candidate forecasting methods using 2015-2017 as training
    and Jan-Jun 2018 as test period.
    """

    methods = [
        (
            "Naive mean",
            lambda x: float(np.mean(x)) if len(x) else 0.0,
        ),
        (
            "Moving average 28d",
            lambda x: float(np.mean(x[-28:])) if len(x) else 0.0,
        ),
        (
            "Moving average 91d",
            lambda x: float(np.mean(x[-91:])) if len(x) else 0.0,
        ),
        (
            "SES alpha 0.2",
            lambda x: ses_forecast(x, 0.2),
        ),
        (
            "Croston alpha 0.1",
            lambda x: croston_forecast(x, 0.1),
        ),
    ]

    rows = []

    train_mask = (
        (daily["Date"] >= TRAIN_START)
        & (daily["Date"] <= TRAIN_END)
    )

    test_mask = (
        (daily["Date"] >= TEST_START)
        & (daily["Date"] <= TEST_END)
    )

    n_test_days = daily.loc[test_mask, "Date"].nunique()

    for sku, g in daily.groupby("SKU", sort=False):
        train = g.loc[train_mask, "Demand"].to_numpy(dtype=float)
        actual_test = float(g.loc[test_mask, "Demand"].sum())

        for method, fn in methods:
            forecast_daily = float(fn(train))
            forecast_test = forecast_daily * n_test_days

            rows.append(
                ForecastResult(
                    method=method,
                    sku=sku,
                    forecast_daily=forecast_daily,
                    actual_test=actual_test,
                    forecast_test=forecast_test,
                    abs_error=abs(actual_test - forecast_test),
                    squared_error=(actual_test - forecast_test) ** 2,
                ).__dict__
            )

    sku_method = pd.DataFrame(rows)

    method_eval = (
        sku_method.groupby("method", as_index=False)
        .agg(
            actual_test=("actual_test", "sum"),
            forecast_test=("forecast_test", "sum"),
            abs_error=("abs_error", "sum"),
            squared_error=("squared_error", "mean"),
        )
    )

    method_eval["WAPE"] = (
        method_eval["abs_error"]
        / method_eval["actual_test"].replace(0, np.nan)
    )

    method_eval["RMSE_per_SKU"] = np.sqrt(method_eval["squared_error"])
    method_eval = method_eval.sort_values("WAPE")

    sku_best = (
        sku_method.sort_values(["sku", "abs_error"])
        .groupby("sku", as_index=False)
        .first()
        .rename(columns={"method": "SelectedMethod"})
    )

    return method_eval, sku_best

In [16]:
method_eval, sku_best = evaluate_forecasts(daily)

display(method_eval)
display(sku_best.head())

,method,actual_test,forecast_test,abs_error,squared_error,WAPE,RMSE_per_SKU
3,Naive mean,20676816.0,1.966532e+07,9.363361e+06,2.212659e+08,0.452843,14875.010246
2,Moving average 91d,20676816.0,2.406338e+07,1.067550e+07,1.701369e+08,0.516303,13043.655049
4,SES alpha 0.2,20676816.0,2.686439e+07,1.319168e+07,5.520551e+08,0.637994,23495.852242
1,Moving average 28d,20676816.0,2.718853e+07,1.363608e+07,5.467988e+08,0.659486,23383.728916
0,Croston alpha 0.1,20676816.0,2.782439e+07,1.394885e+07,5.843285e+08,0.674613,24172.886965


,sku,SelectedMethod,forecast_daily,actual_test,forecast_test,abs_error,squared_error
0,1111,Naive mean,1.197993,92.0,216.836679,124.836679,1.558420e+04
1,1126,Naive mean,64.517336,3940.0,11677.637774,7737.637774,5.987104e+07
2,1127,Moving average 28d,94.571429,16478.0,17117.428571,639.428571,4.088689e+05
3,1129,SES alpha 0.2,30.002078,5969.0,5430.376205,538.623795,2.901156e+05
4,1131,Croston alpha 0.1,29.153983,5319.0,5276.870835,42.129165,1.774867e+03


In [17]:
def standard_normal_loss(z: float) -> float:
    """
    Standard normal loss function:

        L(z) = phi(z) - z * (1 - Phi(z))

    This is used for the expected shortage of normally distributed
    lead-time demand.
    """

    phi = np.exp(-0.5 * z**2) / np.sqrt(2 * np.pi)
    Phi = NORMAL.cdf(z)

    return phi - z * (1 - Phi)


def reorder_point_for_fill_rate(
    mu_daily: float,
    sigma_daily: float,
    lead_time: int,
    q: float,
    target_fill_rate: float,
) -> int:
    """
    Calculates reorder point R for an (R,Q) system using fill-rate logic.

    Lead-time demand:
        mu_L = mu_daily * L
        sigma_L = sigma_daily * sqrt(L)

    Fill-rate approximation:
        beta = 1 - Expected shortage per cycle / Q

    Therefore:
        Expected shortage per cycle = (1 - beta) * Q

    Under normal lead-time demand:
        Expected shortage = sigma_L * L(z)

    where L(z) is the standard normal loss function.
    """

    lead_time = max(int(lead_time), 1)
    q = max(float(q), 1.0)

    target_fill_rate = min(max(target_fill_rate, 0.0001), 0.999999)

    mu_l = mu_daily * lead_time
    sigma_l = sigma_daily * np.sqrt(lead_time)

    if sigma_l <= 1e-12:
        return int(np.ceil(mu_l))

    target_shortage = (1 - target_fill_rate) * q

    # If target shortage is very large, R can be below mean.
    # If target shortage is very small, R must be far above mean.
    low_z = -8.0
    high_z = 8.0

    for _ in range(80):
        mid_z = (low_z + high_z) / 2
        shortage = sigma_l * standard_normal_loss(mid_z)

        if shortage > target_shortage:
            # Too much shortage, increase R by increasing z.
            low_z = mid_z
        else:
            high_z = mid_z

    z = high_z
    r = mu_l + z * sigma_l

    return max(0, int(np.ceil(r)))

In [19]:
reorder_point_for_fill_rate(
    mu_daily=10,
    sigma_daily=4,
    lead_time=7,
    q=100,
    target_fill_rate=0.98,
)

76

In [20]:
def simulate_rq(
    test_series: pd.Series,
    r: int,
    q: float,
    lead_time: int,
) -> dict:
    """
    Simulates a daily continuous-review (R,Q) inventory system.

    Inventory position:
        inventory position = on hand + on order

    Order rule:
        if inventory position <= R, order Q units

    Fill rate:
        shipped demand / total demand
    """

    q = max(int(np.ceil(q)), 1)
    lead_time = max(int(lead_time), 1)

    on_hand = int(r + q)
    on_order = 0
    pipeline = {}

    demand_total = 0.0
    shipped_total = 0.0

    inventory_position_sum = 0.0
    on_hand_sum = 0.0
    order_count = 0

    values = test_series.to_numpy(dtype=float)

    for day, demand in enumerate(values):

        if day in pipeline:
            receipt = pipeline.pop(day)
            on_hand += receipt
            on_order -= receipt

        demand_total += demand

        shipped = min(on_hand, demand)
        shipped_total += shipped
        on_hand -= shipped

        inventory_position = on_hand + on_order

        while inventory_position <= r:
            arrival_day = day + lead_time
            pipeline[arrival_day] = pipeline.get(arrival_day, 0) + q

            on_order += q
            inventory_position += q
            order_count += 1

        inventory_position_sum += inventory_position
        on_hand_sum += on_hand

    fill_rate = (
        shipped_total / demand_total
        if demand_total > 0
        else 1.0
    )

    return {
        "FillRate": fill_rate,
        "Demand": demand_total,
        "Shipped": shipped_total,
        "LostSales": demand_total - shipped_total,
        "AvgOnHand": on_hand_sum / len(values),
        "AvgInventoryPosition": inventory_position_sum / len(values),
        "OrderCount": order_count,
    }

In [21]:
one_sku = daily["SKU"].iloc[0]

test_series = (
    daily[
        (daily["SKU"] == one_sku)
        & (daily["Date"] >= TEST_START)
        & (daily["Date"] <= TEST_END)
    ]
    .set_index("Date")["Demand"]
)

simulate_rq(
    test_series=test_series,
    r=10,
    q=5,
    lead_time=7,
)

{'FillRate': np.float64(0.6086956521739131),
 'Demand': np.float64(92.0),
 'Shipped': np.float64(56.0),
 'LostSales': np.float64(36.0),
 'AvgOnHand': np.float64(11.32596685082873),
 'AvgInventoryPosition': np.float64(13.453038674033149),
 'OrderCount': 11}

In [36]:
def build_policy_table(
    article: pd.DataFrame,
    daily: pd.DataFrame,
    sku_best: pd.DataFrame,
    min_fill_floor: float,
    target_total_fill: float,
) -> pd.DataFrame:
    """
    Builds an (R,Q) policy for all SKUs.

    This version directly calibrates the reorder points R in the simulation
    until the total fill rate reaches the required aggregate target.

    It is more robust than relying only on the theoretical normal approximation,
    especially when 2018 demand is higher than the 2015-2017 forecast.
    """

    train_mask = (
        (daily["Date"] >= TRAIN_START)
        & (daily["Date"] <= TRAIN_END)
    )

    test_mask = (
        (daily["Date"] >= TEST_START)
        & (daily["Date"] <= TEST_END)
    )

    # Historical demand statistics from 2015-2017
    stats = (
        daily.loc[train_mask]
        .groupby("SKU", as_index=False)
        .agg(
            MeanDailyDemand=("Demand", "mean"),
            StdDailyDemand=("Demand", "std"),
        )
    )

    stats["StdDailyDemand"] = stats["StdDailyDemand"].fillna(0)

    base = (
        article[["SKU", "Q", "LeadTime", "UnitPrice"]]
        .merge(stats, on="SKU", how="left")
        .merge(
            sku_best[
                ["sku", "SelectedMethod", "forecast_daily"]
            ].rename(
                columns={
                    "sku": "SKU",
                    "forecast_daily": "ForecastDailyDemand",
                }
            ),
            on="SKU",
            how="left",
        )
    )

    base["MeanDailyDemand"] = base["MeanDailyDemand"].fillna(0)
    base["StdDailyDemand"] = base["StdDailyDemand"].fillna(0)

    base["ForecastDailyDemand"] = base["ForecastDailyDemand"].fillna(
        base["MeanDailyDemand"]
    )

    # Important correction:
    # Use the larger of selected forecast and historical mean.
    # This avoids very low reorder points when a forecasting method underestimates demand.
    base["DemandEstimateDaily"] = base[
        ["ForecastDailyDemand", "MeanDailyDemand"]
    ].max(axis=1)

    sku_series = {
        sku: g.loc[test_mask, ["Date", "Demand"]]
        .set_index("Date")["Demand"]
        for sku, g in daily.groupby("SKU", sort=False)
    }

    total_test_demand = sum(series.sum() for series in sku_series.values())

    if total_test_demand <= 0:
        raise ValueError(
            "Total demand in Jan-Jun 2018 is zero. "
            "Check demand date parsing before running the inventory model."
        )

    def evaluate(extra_safety_factor: float) -> pd.DataFrame:
        """
        extra_safety_factor is a common multiplier added to the safety stock.

        R = mean lead-time demand + z_floor * std lead-time demand
            + extra_safety_factor * std lead-time demand

        The binary search increases extra_safety_factor until total fill rate
        reaches the 98% target.
        """

        rows = []

        # Minimum SKU floor converted to a z-value.
        # 50% -> z = 0
        # 60% -> z ≈ 0.25
        # 70% -> z ≈ 0.52
        # 80% -> z ≈ 0.84
        z_floor = NORMAL.inv_cdf(min_fill_floor)

        for row in base.itertuples(index=False):

            lead_time = max(int(row.LeadTime), 1)

            mu_l = float(row.DemandEstimateDaily) * lead_time
            sigma_l = float(row.StdDailyDemand) * np.sqrt(lead_time)

            r = mu_l + z_floor * sigma_l + extra_safety_factor * sigma_l
            r = max(0, int(np.ceil(r)))

            sim = simulate_rq(
                test_series=sku_series[row.SKU],
                r=r,
                q=float(row.Q),
                lead_time=lead_time,
            )

            rows.append(
                {
                    "SKU": row.SKU,
                    "SelectedMethod": row.SelectedMethod,
                    "ForecastDailyDemand": row.ForecastDailyDemand,
                    "MeanDailyDemand": row.MeanDailyDemand,
                    "DemandEstimateDaily": row.DemandEstimateDaily,
                    "StdDailyDemand": row.StdDailyDemand,
                    "LeadTime": row.LeadTime,
                    "Q": row.Q,
                    "R": r,
                    "MinFillFloor": min_fill_floor,
                    "ExtraSafetyFactor": extra_safety_factor,
                    **sim,
                    "UnitPrice": row.UnitPrice,
                    "AvgInventoryValue": sim["AvgOnHand"] * row.UnitPrice,
                }
            )

        return pd.DataFrame(rows)

    # ------------------------------------------------------------
    # Binary search for the required extra safety stock
    # ------------------------------------------------------------

    low = 0.0
    high = 1.0

    # Increase upper bound until the target fill rate is reachable.
    while True:
        candidate = evaluate(high)

        candidate_total_fill = (
            candidate["Shipped"].sum()
            / candidate["Demand"].sum()
        )

        if candidate_total_fill >= target_total_fill:
            break

        high *= 2

        if high > 100:
            raise ValueError(
                "Could not reach the target fill rate even with very large "
                "safety stock. Check Q, lead times, and demand data."
            )

    best = candidate

    for _ in range(40):
        mid = (low + high) / 2

        candidate = evaluate(mid)

        candidate_total_fill = (
            candidate["Shipped"].sum()
            / candidate["Demand"].sum()
        )

        if candidate_total_fill >= target_total_fill:
            best = candidate
            high = mid
        else:
            low = mid

    final_fill = best["Shipped"].sum() / best["Demand"].sum()

    print(
        f"Minimum floor {min_fill_floor:.0%}: "
        f"extra safety factor = {high:.4f}, "
        f"simulated total fill rate = {final_fill:.4%}"
    )

    return best

In [37]:
def summarize_inventory(
    policy_tables: dict[float, pd.DataFrame],
) -> pd.DataFrame:
    rows = []

    for floor, table in policy_tables.items():
        rows.append(
            {
                "Minimum SKU Fill-Rate Floor": floor,
                "Total Demand": table["Demand"].sum(),
                "Total Shipped": table["Shipped"].sum(),
                "Total Lost Sales": table["LostSales"].sum(),
                "Total Fill Rate": (
                    table["Shipped"].sum()
                    / table["Demand"].sum()
                ),
                "Average SKU Fill Rate": table["FillRate"].mean(),
                "Average On Hand Units": table["AvgOnHand"].sum(),
                "Average Inventory Value": table[
                    "AvgInventoryValue"
                ].sum(),
                "Total Orders": table["OrderCount"].sum(),
                "Average R": table["R"].mean(),
                "Average Q": table["Q"].mean(),
            }
        )

    return pd.DataFrame(rows)

In [38]:
policies = {
    floor: build_policy_table(
        article=article,
        daily=daily,
        sku_best=sku_best,
        min_fill_floor=floor,
        target_total_fill=TOTAL_FILL_RATE_TARGET,
    )
    for floor in MIN_FILL_RATE_GRID
}

inventory_summary = summarize_inventory(policies)

display(inventory_summary)

Minimum floor 50%: extra safety factor = 11.1256, simulated total fill rate = 98.0000%
Minimum floor 60%: extra safety factor = 10.8723, simulated total fill rate = 98.0000%
Minimum floor 70%: extra safety factor = 10.6012, simulated total fill rate = 98.0000%
Minimum floor 80%: extra safety factor = 10.2840, simulated total fill rate = 98.0000%


,Minimum SKU Fill-Rate Floor,Total Demand,Total Shipped,Total Lost Sales,Total Fill Rate,Average SKU Fill Rate,Average On Hand Units,Average Inventory Value,Total Orders,Average R,Average Q
0,0.5,20676816.0,20263280.0,413536.0,0.98,0.989096,8.414369e+06,5.923219e+07,1173017,5809.414974,790.698201
1,0.6,20676816.0,20263280.0,413536.0,0.98,0.989096,8.414369e+06,5.923219e+07,1173017,5809.414974,790.698201
2,0.7,20676816.0,20263280.0,413536.0,0.98,0.989096,8.414369e+06,5.923219e+07,1173017,5809.414974,790.698201
3,0.8,20676816.0,20263280.0,413536.0,0.98,0.989096,8.414369e+06,5.923219e+07,1173017,5809.414974,790.698201


In [39]:
base_extra_safety = policies[0.50]["ExtraSafetyFactor"].iloc[0]

print("Base extra safety factor from 50% floor:", base_extra_safety)

Base extra safety factor from 50% floor: 11.125615466284216


In [40]:
def build_policy_table_fixed_safety(
    article: pd.DataFrame,
    daily: pd.DataFrame,
    sku_best: pd.DataFrame,
    min_fill_floor: float,
    extra_safety_factor: float,
) -> pd.DataFrame:
    """
    Builds an (R,Q) policy using a fixed extra safety factor.

    This is useful for sensitivity analysis:
    first calibrate to 98% at the 50% floor, then keep the safety factor fixed
    and vary the minimum SKU fill-rate floor.
    """

    train_mask = (
        (daily["Date"] >= TRAIN_START)
        & (daily["Date"] <= TRAIN_END)
    )

    test_mask = (
        (daily["Date"] >= TEST_START)
        & (daily["Date"] <= TEST_END)
    )

    stats = (
        daily.loc[train_mask]
        .groupby("SKU", as_index=False)
        .agg(
            MeanDailyDemand=("Demand", "mean"),
            StdDailyDemand=("Demand", "std"),
        )
    )

    stats["StdDailyDemand"] = stats["StdDailyDemand"].fillna(0)

    base = (
        article[["SKU", "Q", "LeadTime", "UnitPrice"]]
        .merge(stats, on="SKU", how="left")
        .merge(
            sku_best[
                ["sku", "SelectedMethod", "forecast_daily"]
            ].rename(
                columns={
                    "sku": "SKU",
                    "forecast_daily": "ForecastDailyDemand",
                }
            ),
            on="SKU",
            how="left",
        )
    )

    base["MeanDailyDemand"] = base["MeanDailyDemand"].fillna(0)
    base["StdDailyDemand"] = base["StdDailyDemand"].fillna(0)

    base["ForecastDailyDemand"] = base["ForecastDailyDemand"].fillna(
        base["MeanDailyDemand"]
    )

    base["DemandEstimateDaily"] = base[
        ["ForecastDailyDemand", "MeanDailyDemand"]
    ].max(axis=1)

    sku_series = {
        sku: g.loc[test_mask, ["Date", "Demand"]]
        .set_index("Date")["Demand"]
        for sku, g in daily.groupby("SKU", sort=False)
    }

    rows = []

    z_floor = NORMAL.inv_cdf(min_fill_floor)

    for row in base.itertuples(index=False):
        lead_time = max(int(row.LeadTime), 1)

        mu_l = float(row.DemandEstimateDaily) * lead_time
        sigma_l = float(row.StdDailyDemand) * np.sqrt(lead_time)

        r = (
            mu_l
            + z_floor * sigma_l
            + extra_safety_factor * sigma_l
        )

        r = max(0, int(np.ceil(r)))

        sim = simulate_rq(
            test_series=sku_series[row.SKU],
            r=r,
            q=float(row.Q),
            lead_time=lead_time,
        )

        rows.append(
            {
                "SKU": row.SKU,
                "SelectedMethod": row.SelectedMethod,
                "ForecastDailyDemand": row.ForecastDailyDemand,
                "MeanDailyDemand": row.MeanDailyDemand,
                "DemandEstimateDaily": row.DemandEstimateDaily,
                "StdDailyDemand": row.StdDailyDemand,
                "LeadTime": row.LeadTime,
                "Q": row.Q,
                "R": r,
                "MinFillFloor": min_fill_floor,
                "ExtraSafetyFactor": extra_safety_factor,
                **sim,
                "UnitPrice": row.UnitPrice,
                "AvgInventoryValue": sim["AvgOnHand"] * row.UnitPrice,
            }
        )

    return pd.DataFrame(rows)

In [41]:
sensitivity_policies = {
    floor: build_policy_table_fixed_safety(
        article=article,
        daily=daily,
        sku_best=sku_best,
        min_fill_floor=floor,
        extra_safety_factor=base_extra_safety,
    )
    for floor in MIN_FILL_RATE_GRID
}

sensitivity_summary = summarize_inventory(sensitivity_policies)

display(sensitivity_summary)

,Minimum SKU Fill-Rate Floor,Total Demand,Total Shipped,Total Lost Sales,Total Fill Rate,Average SKU Fill Rate,Average On Hand Units,Average Inventory Value,Total Orders,Average R,Average Q
0,0.5,20676816.0,20263280.0,413536.0,0.980000,0.989096,8.414369e+06,5.923219e+07,1173017,5809.414974,790.698201
1,0.6,20676816.0,20278090.0,398726.0,0.980716,0.989435,8.558373e+06,6.028726e+07,1175198,5894.039466,790.698201
2,0.7,20676816.0,20292738.0,384078.0,0.981425,0.989803,8.712511e+06,6.141656e+07,1177426,5984.586187,790.698201
3,0.8,20676816.0,20309391.0,367425.0,0.982230,0.990199,8.893141e+06,6.273960e+07,1180029,6090.543239,790.698201


In [42]:
assumptions = pd.DataFrame(
    {
        "Item": [
            "Training period",
            "Evaluation period",
            "Forecast selection",
            "Total fill-rate target",
            "Minimum SKU fill-rate floors",
            "R calculation",
            "Q calculation",
            "Inventory simulation",
        ],
        "Value": [
            f"{TRAIN_START} to {TRAIN_END}",
            f"{TEST_START} to {TEST_END}",
            (
                "Lowest absolute test error per SKU among mean, "
                "moving-average, SES and Croston methods"
            ),
            TOTAL_FILL_RATE_TARGET,
            ", ".join(f"{x:.0%}" for x in MIN_FILL_RATE_GRID),
            (
                "Normal lead-time demand approximation with binary "
                "search calibration to aggregate fill-rate target"
            ),
            "MOQ from ArticleData",
            (
                "Daily continuous-review lost-sales simulation with "
                "pipeline receipts after lead time"
            ),
        ],
    }
)

display(assumptions)

,Item,Value
0,Training period,2015-01-01 to 2017-12-31
1,Evaluation period,2018-01-01 to 2018-06-30
2,Forecast selection,"Lowest absolute test error per SKU among mean,..."
3,Total fill-rate target,0.98
4,Minimum SKU fill-rate floors,"50%, 60%, 70%, 80%"
5,R calculation,Normal lead-time demand approximation with bin...
6,Q calculation,MOQ from ArticleData
7,Inventory simulation,Daily continuous-review lost-sales simulation ...
